# FoldCraft

This notebook uses the tested command-line pipeline in this checkout. Residue
mapping, contact-map construction and MPNN selections share the CLI helpers.

Use a prepared FoldCraft environment with its model weights for the inference
cell. Configuration and input preflight run on CPU without GPU packages. See
README.md for installation; GPU compatibility must be checked on the intended
machine. In Colab, make this checkout available and set `repo_root` below.

Selections use original PDB numbering, inclusive at both ends. Prepare cropped
PDB files explicitly before this notebook; incomplete backbones, numbering gaps
and insertion codes fail preflight. Use a new output directory for each run.


In [ ]:
# Set repo_root to this checkout when running from another directory or Colab.
from pathlib import Path
import subprocess
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / "FoldCraft.py").is_file():
    raise ValueError("Set repo_root to your FoldCraft checkout before continuing")
output_folder = repo_root / "runs" / "notebook-fold"
target_template = repo_root / "examples" / "targets" / "pd-l1-1.pdb"
target_chain = "A"
target_hotspots = "30-34,50-54,69-76"  # original PDB residue numbers, inclusive
num_designs = 1
mpnn_samples = 5
design_stages = "100,100,20"
binder_template = repo_root / "examples" / "templates" / "1qys1.pdb"
binder_chain = "A"
binder_hotspots = "26-40,58-71"
binder_mask = ""


In [ ]:
command = [sys.executable, str(repo_root / "FoldCraft.py"),
           "--output_folder", str(output_folder),
           "--target_template", str(target_template), "--target_chain", target_chain,
           "--target_hotspots", target_hotspots,
           "--num_designs", str(num_designs), "--mpnn_samples", str(mpnn_samples),
           "--design_stages", design_stages]
command += ["--binder_template", str(binder_template), "--binder_chain", binder_chain,
            "--binder_hotspots", binder_hotspots, "--binder_mask", binder_mask]
subprocess.run(command + ["--preflight_only"], cwd=repo_root, check=True)


The next cell runs GPU inference. It can take a substantial amount of time.
The current validation protocol is explicitly model_1_ptm with three recycles.


In [ ]:
subprocess.run(command, cwd=repo_root, check=True)


In [ ]:
import pandas as pd
results_path = output_folder / "results.csv"
if results_path.exists():
    display(pd.read_csv(results_path))
